# Notebook 17
## DNA Nucleotide Transformer Embedding Extraction

Extracts pretrained transformer embeddings for DNA sequences using the
**Nucleotide Transformer** (`InstaDeepAI/nucleotide-transformer-500m-human-ref`),
a 500M-parameter BERT-style model pretrained on the human reference genome.

### Pipeline
```
DNA sequence (200 bp)
-> NT tokenizer  (6-mer BPE, special [CLS] token prepended)
-> Transformer encoder (500M parameters)
-> Mean pooling over non-padding token hidden states
-> Fixed-length embedding vector (1024-dim)
-> Save as .npy
```

### Key differences from DNABERT-2 (notebook 15)
- NT uses a **6-mer k-mer tokenizer** -- sequences must be passed as raw strings;
  the tokenizer handles k-mer splitting internally.
- NT's hidden dimension is **1024** vs DNABERT-2's 768.
- NT does not use ALiBi; no `bert_layers.py` patches are needed.
- NT uses standard HuggingFace `AutoTokenizer` / `AutoModel` with no
  `trust_remote_code` required.
- Mean pooling excludes the `[CLS]` token (index 0) to match the approach
  recommended in the NT paper for sequence-level tasks.

### Model choice
`nucleotide-transformer-500m-human-ref` is pretrained specifically on the human
reference genome (GRCh38), making it the most directly relevant NT variant for
this promoter classification task (human sequences, GRCh38).

### Outputs
- `data/processed/dna_nt_embeddings_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `data/processed/dna_nt_labels_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `data/processed/dna_nt_ids_len{L}_pos{N_POS}_neg{N_NEG}.npy`
- `reports/dna_nt_embeddings_summary.json`

## 0) Installation note

```bash
pip install transformers torch numpy pandas pyyaml tqdm
```

Model card: https://huggingface.co/InstaDeepAI/nucleotide-transformer-500m-human-ref

No `trust_remote_code=True` needed -- NT uses standard HuggingFace ESMModel
architecture registered in transformers core.

## 1) Imports

In [1]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings('ignore')

print('torch:          ', torch.__version__)
print('CUDA available: ', torch.cuda.is_available())


torch:           2.10.0+cu128
CUDA available:  True


## 2) Paths, config, seed

In [2]:
ROOT      = Path.cwd().parents[0]
PROCESSED = ROOT / 'data' / 'processed'
REPORTS   = ROOT / 'reports'
CONFIGS   = ROOT / 'configs'

for p in [PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

with open(CONFIGS / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

SEED  = int(cfg['project']['random_seed'])
L     = int(cfg['dna']['seq_length_bp'])
N_POS = int(cfg['dna']['n_pos'])
N_NEG = int(cfg['dna']['n_neg'])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'SEED={SEED}  L={L}  N_POS={N_POS}  N_NEG={N_NEG}')


SEED=42  L=200  N_POS=2000  N_NEG=2000


## 3) Device selection

In [3]:
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print('Using device:', DEVICE)


Using device: cuda


## 4) Load the processed DNA dataset

Same CSV as all prior DNA notebooks. Rows embedded in original order;
notebook 18 will apply the split.

In [4]:
in_csv = PROCESSED / f'dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv'
assert in_csv.exists(), f'Not found: {in_csv} -- run 03_dna_ingest.ipynb first'

df = pd.read_csv(in_csv)

required = ['region_id', 'sequence', 'label']
assert all(c in df.columns for c in required)
assert df['sequence'].isna().sum() == 0
assert (df['sequence'].str.len() != L).sum() == 0

print('Shape:', df.shape)
print(df['label'].value_counts().sort_index())
print(df.head(3))


Shape: (4000, 6)
label
0    2000
1    2000
Name: count, dtype: int64
  region_id chrom      start        end  \
0    prom_0     4   88592334   88592533   
1    prom_1     5  120078977  120079176   
2    prom_2     8    9371095    9371294   

                                            sequence  label  
0  TGCCCGCGGACCTTGCCGCCCCGCCTCCAGCCCGTGCCACGGCGGC...      1  
1  AGAAAGAAGGAGAAAAAACGGCTCAAAGAAGAGTTGATGGCTGGGA...      1  
2  TATGGAAAGCTTCACAAATTTGCACGGCATCTTTGTGCAGGGCCGT...      1  


## 5) Load NT tokenizer and model

### Model selection rationale
`nucleotide-transformer-500m-human-ref` is pretrained on the human reference
genome (GRCh38), the same build used to generate our promoter sequences.
This is the most directly relevant NT variant for this task.

Other available variants for reference:
- `nucleotide-transformer-500m-1000g` -- pretrained on 1000 Genomes Project
- `nucleotide-transformer-2.5b-multi-species` -- larger, multi-species

### Tokenizer behaviour
NT uses a 6-mer tokenizer. A 200 bp sequence produces roughly 33-34 tokens
(200 / 6 = 33.3) plus the `[CLS]` token prepended at position 0.
Sequences whose length is not a multiple of 6 are handled with padding tokens
at the end. Feed raw uppercase strings -- no manual k-mer splitting needed.

### Memory
The 500M model requires ~2 GB GPU memory at fp32. `batch_size=16` is safe
on a 16 GB GPU; use 32 on A100/H100.

In [5]:
MODEL_NAME = 'InstaDeepAI/nucleotide-transformer-500m-human-ref'

print(f'Loading tokenizer from: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'  vocab_size:    {tokenizer.vocab_size}')
print(f'  cls_token_id:  {tokenizer.cls_token_id}')
print(f'  pad_token_id:  {tokenizer.pad_token_id}')
print(f'  model_max_len: {tokenizer.model_max_length}')

print(f'\nLoading model from: {MODEL_NAME}')
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Model dtype:      {next(model.parameters()).dtype}')


Loading tokenizer from: InstaDeepAI/nucleotide-transformer-500m-human-ref
  vocab_size:    4107
  cls_token_id:  3
  pad_token_id:  1
  model_max_len: 1000

Loading model from: InstaDeepAI/nucleotide-transformer-500m-human-ref


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 480,438,241
Model dtype:      torch.float32


## 6) Tokenization check

Verify token count and structure for one example sequence.
200 bp / 6 = 33.3, so we expect ~34 6-mer tokens + 1 CLS = ~35 tokens total.

In [6]:
example_seq = df['sequence'].iloc[0].upper()

enc = tokenizer(
    example_seq,
    return_tensors='pt',
    padding=False,
    truncation=True,
    max_length=512,
)

print('Sequence length (bp):   ', len(example_seq))
print('input_ids shape:        ', enc['input_ids'].shape)
print('attention_mask sum:     ', enc['attention_mask'].sum().item())

tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].tolist())
print('All tokens:             ', tokens)
print('Token 0 (should be CLS):', tokens[0])


Sequence length (bp):    200
input_ids shape:         torch.Size([1, 36])
attention_mask sum:      36
All tokens:              ['<cls>', 'TGCCCG', 'CGGACC', 'TTGCCG', 'CCCCGC', 'CTCCAG', 'CCCGTG', 'CCACGG', 'CGGCCG', 'CCATTG', 'GCGCGG', 'GCCCAT', 'CCCAGA', 'ACGGCG', 'CCCATT', 'GGCCCG', 'GTGCGA', 'AGCCAT', 'TCACCC', 'AGCGCA', 'TTCCGC', 'CGCCCG', 'CAGGCT', 'TGCGGG', 'GAAACT', 'TCCTTA', 'TTATTG', 'TGACGC', 'CGAAAA', 'CGGAGA', 'AACCCC', 'GGGTCC', 'GGCGAG', 'AGGGGC', 'T', 'G']
Token 0 (should be CLS): <cls>


## 7) Embedding extraction function

### Pooling strategy
We **exclude token 0** (`[CLS]`) from the mean pool and average over the
remaining real (non-padding) tokens. The NT paper reports that mean pooling
over the body tokens (excluding CLS) outperforms CLS-only for sequence
classification tasks.

Concretely, for batch element `i` with hidden states `H` (T, D) and
attention mask `m` (T):
```
# zero out CLS position
m[0] = 0
embedding[i] = sum(H[t] * m[t]) / sum(m[t])
```

This is different from DNABERT-2 (where CLS was included) and ESM-2
(where CLS/EOS were excluded by index slicing). We use mask zeroing here
because it is agnostic to token layout and batch padding.

In [7]:
def mean_pool_no_cls(hidden_states, attention_mask):
    """
    Mean pool over real tokens, excluding position 0 (CLS).

    Args:
        hidden_states:  (batch, seq_len, hidden_dim)
        attention_mask: (batch, seq_len)  -- 1 real, 0 padding

    Returns:
        pooled: (batch, hidden_dim)
    """
    # Clone mask so we don't modify the original
    mask = attention_mask.clone().float()
    # Zero out CLS token (index 0) in every sequence
    mask[:, 0] = 0.0
    mask_expanded = mask.unsqueeze(-1)                          # (B, T, 1)
    sum_hidden    = (hidden_states * mask_expanded).sum(dim=1)  # (B, D)
    sum_mask      = mask_expanded.sum(dim=1).clamp(min=1e-9)    # (B, 1)
    return sum_hidden / sum_mask                                 # (B, D)


def extract_nt_embeddings(
    sequences,
    tokenizer,
    model,
    device,
    batch_size=16,
    max_length=512,
):
    """
    Extract NT mean-pooled embeddings (CLS excluded) for a list of DNA sequences.

    Args:
        sequences:   List of raw uppercase DNA strings.
        tokenizer:   HuggingFace tokenizer for NT.
        model:       Loaded NT AutoModel in eval mode.
        device:      Torch device string.
        batch_size:  Sequences per forward pass (16 safe on 16 GB GPU).
        max_length:  Hard token limit (200 bp -> ~35 tokens, well within 512).

    Returns:
        embeddings: np.ndarray of shape (len(sequences), hidden_dim).
    """
    all_embeddings = []
    model.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size), desc='Extracting NT embeddings'):
            batch_seqs = [s.upper() for s in sequences[i : i + batch_size]]

            encoded = tokenizer(
                batch_seqs,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            input_ids      = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            # last_hidden_state: (B, T, D)
            hidden_states = outputs.last_hidden_state

            pooled = mean_pool_no_cls(hidden_states, attention_mask)
            all_embeddings.append(pooled.cpu().float().numpy())

    return np.concatenate(all_embeddings, axis=0)


## 8) Run extraction

Expected runtime on A100 (40 GB) with `batch_size=32`: ~2-3 min for 4000
sequences (500M model is ~4x larger than DNABERT-2's 117M).
Reduce `BATCH_SIZE` if you get CUDA OOM.

In [8]:
# 16 GB GPU -> 16;  40 GB GPU (A100) -> 32
BATCH_SIZE = 16

sequences = df['sequence'].tolist()

embeddings = extract_nt_embeddings(
    sequences=sequences,
    tokenizer=tokenizer,
    model=model,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_length=512,
)

print('Embeddings shape:', embeddings.shape)
print('Dtype:           ', embeddings.dtype)


Extracting NT embeddings:   0%|          | 0/250 [00:00<?, ?it/s]

Embeddings shape: (4000, 1280)
Dtype:            float32


## 9) Sanity checks

In [9]:
assert embeddings.shape[0] == len(df), 'Row count mismatch'
assert not np.isnan(embeddings).any(), 'NaN in embeddings'
assert not np.isinf(embeddings).any(), 'Inf in embeddings'

print('Shape:    ', embeddings.shape)
print('Dtype:    ', embeddings.dtype)
print(f'Mean:     {embeddings.mean():.6f}')
print(f'Std:      {embeddings.std():.6f}')
print(f'Min:      {embeddings.min():.6f}')
print(f'Max:      {embeddings.max():.6f}')
print('Any NaN:  ', np.isnan(embeddings).any())
print('Any Inf:  ', np.isinf(embeddings).any())


Shape:     (4000, 1280)
Dtype:     float32
Mean:     -0.000780
Std:      0.825656
Min:      -8.960460
Max:      17.628508
Any NaN:   False
Any Inf:   False


## 10) Save embeddings, labels, and region IDs

Three parallel arrays in original row order.
Notebook 18 will apply `train_test_split(random_state=SEED, test_size=0.2,
stratify=y)` -- identical to all prior DNA notebooks.

In [10]:
labels = df['label'].astype(int).values
ids    = df['region_id'].astype(str).values

sfx         = f'len{L}_pos{N_POS}_neg{N_NEG}'
emb_path    = PROCESSED / f'dna_nt_embeddings_{sfx}.npy'
labels_path = PROCESSED / f'dna_nt_labels_{sfx}.npy'
ids_path    = PROCESSED / f'dna_nt_ids_{sfx}.npy'

np.save(emb_path,    embeddings.astype(np.float32))
np.save(labels_path, labels)
np.save(ids_path,    ids)

print('Saved:', emb_path)
print('Saved:', labels_path)
print('Saved:', ids_path)


Saved: /home/dpratapa/Capstone/data/processed/dna_nt_embeddings_len200_pos2000_neg2000.npy
Saved: /home/dpratapa/Capstone/data/processed/dna_nt_labels_len200_pos2000_neg2000.npy
Saved: /home/dpratapa/Capstone/data/processed/dna_nt_ids_len200_pos2000_neg2000.npy


## 11) Reload verification

In [11]:
X_check  = np.load(emb_path)
y_check  = np.load(labels_path)
id_check = np.load(ids_path, allow_pickle=True)

assert X_check.shape  == embeddings.shape
assert y_check.shape  == labels.shape
assert id_check.shape == ids.shape
assert not np.isnan(X_check).any()
assert (y_check == labels).all()

print('Reload check passed.')
print('Embedding file shape:', X_check.shape)
print('Labels file shape:   ', y_check.shape)
print('IDs file shape:      ', id_check.shape)


Reload check passed.
Embedding file shape: (4000, 1280)
Labels file shape:    (4000,)
IDs file shape:       (4000,)


## 12) Save summary JSON

In [12]:
label_counts = {int(k): int(v) for k, v in zip(*np.unique(labels, return_counts=True))}

summary = {
    'notebook': '17_dna_nt_embeddings',
    'model': MODEL_NAME,
    'model_size': '500M parameters',
    'pretrain_data': 'human reference genome GRCh38',
    'tokenizer_type': '6-mer k-mer tokenizer',
    'pooling_strategy': 'mean pooling over body tokens (CLS excluded, mask-weighted)',
    'embedding_dim': int(embeddings.shape[1]),
    'n_sequences': int(embeddings.shape[0]),
    'label_counts': label_counts,
    'seq_length_bp': L,
    'max_token_length': 512,
    'batch_size_used': BATCH_SIZE,
    'device': DEVICE,
    'seed': SEED,
    'embedding_stats': {
        'mean': float(embeddings.mean()),
        'std':  float(embeddings.std()),
        'min':  float(embeddings.min()),
        'max':  float(embeddings.max()),
        'any_nan': bool(np.isnan(embeddings).any()),
        'any_inf': bool(np.isinf(embeddings).any()),
    },
    'output_files': {
        'embeddings': str(emb_path),
        'labels':     str(labels_path),
        'ids':        str(ids_path),
    },
    'timestamp': pd.Timestamp.now().isoformat(),
}

summary_path = REPORTS / 'dna_nt_embeddings_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to:', summary_path)
print(json.dumps(summary, indent=2))


Summary saved to: /home/dpratapa/Capstone/reports/dna_nt_embeddings_summary.json
{
  "notebook": "17_dna_nt_embeddings",
  "model": "InstaDeepAI/nucleotide-transformer-500m-human-ref",
  "model_size": "500M parameters",
  "pretrain_data": "human reference genome GRCh38",
  "tokenizer_type": "6-mer k-mer tokenizer",
  "pooling_strategy": "mean pooling over body tokens (CLS excluded, mask-weighted)",
  "embedding_dim": 1280,
  "n_sequences": 4000,
  "label_counts": {
    "0": 2000,
    "1": 2000
  },
  "seq_length_bp": 200,
  "max_token_length": 512,
  "batch_size_used": 16,
  "device": "cuda",
  "seed": 42,
  "embedding_stats": {
    "mean": -0.0007795931887812912,
    "std": 0.8256564736366272,
    "min": -8.96045970916748,
    "max": 17.628507614135742,
    "any_nan": false,
    "any_inf": false
  },
  "output_files": {
    "embeddings": "/home/dpratapa/Capstone/data/processed/dna_nt_embeddings_len200_pos2000_neg2000.npy",
    "labels": "/home/dpratapa/Capstone/data/processed/dna_nt_l

## Next

Proceed to `18_dna_nt_models.ipynb`:
- Loads `dna_nt_embeddings_*.npy` and `dna_nt_labels_*.npy`
- Applies `train_test_split(random_state=SEED, test_size=0.2, stratify=y)`
- Trains: Logistic Regression, SVM, Random Forest, XGBoost
- Reports: accuracy, precision, recall, F1, ROC-AUC
- Saves: CSV + JSON results, model `.pkl` files